# AquaSynex Phase 2.5A: Modeling Dataset Preparation & Integrity Audit

**SIH26146 – AI-Powered Monitoring & Analysis of Bitcoin Transaction Traffic**

This notebook provides a comprehensive architectural and statistical audit of the unified machine learning modeling dataset
(`data/processed/modeling/modeling_dataset.parquet` and DuckDB `modeling_dataset_v1`).

> **Anti-Leakage & Governance Mandate**:
> - **NO machine learning models are trained** in this notebook.
> - **NO model training code or weights** are generated.
> - **46 Canonical Predictive Features** (44 numeric + 2 categorical) form the primary feature space.
> - `hist_cluster_id` is classified strictly as an **IDENTIFIER / GROUPING KEY** and is excluded from ML model features.
> - The 4 graph duplicates (`graph_fan_in/out`, `graph_unique_in/out_addrs`) are relegated to **OPTIONAL ABLATION FEATURES**.
> - Strict chronological ordering and temporal split partitions (70% train, 15% val, 15% test) are validated.

In [1]:
import os
import yaml
import duckdb
import numpy as np
import pandas as pd

# Connect to DuckDB or read directly from Parquet
pq_path = 'data/processed/modeling/modeling_dataset.parquet' if os.path.exists('data/processed/modeling/modeling_dataset.parquet') else '../data/processed/modeling/modeling_dataset.parquet'
con = duckdb.connect()
df_model = con.execute(f"SELECT * FROM read_parquet('{pq_path}')").df()
con.close()

# Load Feature Manifest
manifest_path = 'data/processed/modeling/feature_manifest.yaml' if os.path.exists('data/processed/modeling/feature_manifest.yaml') else '../data/processed/modeling/feature_manifest.yaml'
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = yaml.safe_load(f)

print(f"[*] Loaded Modeling Dataset: {df_model.shape[0]:,} rows x {df_model.shape[1]} columns")
print(f"[*] Primary Canonical Predictive Features: {manifest['dataset_summary']['canonical_predictive_features_count']}")
print(f"    - Canonical Numeric Features:      {manifest['dataset_summary']['canonical_numeric_features_count']}")
print(f"    - Canonical Categorical Features:  {manifest['dataset_summary']['canonical_categorical_features_count']}")
print(f"[*] Optional Ablation Features:         {manifest['dataset_summary']['optional_ablation_features_count']}")
print(f"[*] Identifiers & Grouping Keys:        {manifest['dataset_summary']['identifiers_and_grouping_count']}")

[*] Loaded Modeling Dataset: 10,000 rows x 56 columns
[*] Primary Canonical Predictive Features: 46
    - Canonical Numeric Features:      44
    - Canonical Categorical Features:  2
[*] Optional Ablation Features:         4
[*] Identifiers & Grouping Keys:        3


In [2]:
# Data Quality and Integrity Audit
num_cols = df_model.select_dtypes(include=[np.number]).columns
nan_counts = df_model[num_cols].isna().sum().sum()
inf_counts = np.isinf(df_model[num_cols]).sum().sum()
dup_ids = df_model['transaction_id'].duplicated().sum()

# Temporal ordering verification
timestamps = df_model['timestamp_epoch_sec'].values
is_strictly_ordered = bool(np.all(timestamps[:-1] <= timestamps[1:]))

print('=== DATA QUALITY & INTEGRITY AUDIT ===')
print(f'Total Numeric Columns:          {len(num_cols)}')
print(f'Total Missing (NaN) Values:     {nan_counts}')
print(f'Total Infinite (Inf) Values:    {inf_counts}')
print(f'Duplicate Transaction IDs:      {dup_ids}')
print(f'Monotonic Chronological Order:  {is_strictly_ordered}')

=== DATA QUALITY & INTEGRITY AUDIT ===
Total Numeric Columns:          51
Total Missing (NaN) Values:     0
Total Infinite (Inf) Values:    0
Duplicate Transaction IDs:      0
Monotonic Chronological Order:  True


In [3]:
# Ground-Truth Target Distribution Audit
print('=== TARGET DISTRIBUTION AUDIT ===')
binary_dist = df_model['target_binary'].value_counts(normalize=True).mul(100).round(2)
print('Binary Target Breakdown (0=Benign, 1=Suspicious):')
for val, pct in binary_dist.items():
    cnt = (df_model['target_binary'] == val).sum()
    label_desc = 'Benign' if val == 0 else 'Suspicious'
    print(f'  - Class {val} ({label_desc}): {cnt:,} records ({pct:.2f}%)')

print('\nMulticlass Behavior Scenarios (11 Classes):')
multiclass_dist = df_model['target_multiclass'].value_counts()
for scenario, cnt in multiclass_dist.items():
    print(f'  - {scenario:<22}: {cnt:>5,} ({cnt / len(df_model):.2%})')

=== TARGET DISTRIBUTION AUDIT ===
Binary Target Breakdown (0=Benign, 1=Suspicious):
  - Class 0 (Benign): 5,663 records (56.63%)
  - Class 1 (Suspicious): 4,337 records (43.37%)

Multiclass Behavior Scenarios (11 Classes):
  - normal                : 5,079 (50.79%)
  - transaction_burst     : 1,124 (11.24%)
  - rapid_multihop        :   972 (9.72%)
  - peeling_chain         :   784 (7.84%)
  - coordinated_activity  :   591 (5.91%)
  - benign_high_volume    :   584 (5.84%)
  - high_fan_in           :   223 (2.23%)
  - temporal_anomaly      :   209 (2.09%)
  - high_fan_out          :   205 (2.05%)
  - mixing_like           :   152 (1.52%)
  - amount_anomaly        :    77 (0.77%)


In [4]:
# Chronological Temporal Split Strategy Audit
print('=== TEMPORAL SPLIT PARTITION AUDIT ===')
split_summary = []
for split_name in ['train', 'val', 'test']:
    split_sub = df_model[df_model['temporal_split'] == split_name]
    t_min = split_sub['timestamp_epoch_sec'].min()
    t_max = split_sub['timestamp_epoch_sec'].max()
    susp_pct = split_sub['target_binary'].mean() * 100
    scenario_cnt = split_sub['target_multiclass'].nunique()
    split_summary.append({
        'Partition': split_name,
        'Records': len(split_sub),
        'Share': f'{len(split_sub)/len(df_model):.1%}',
        'Start Epoch': t_min,
        'End Epoch': t_max,
        'Suspicious %': f'{susp_pct:.2f}%',
        'Scenario Classes': scenario_cnt
    })

print(pd.DataFrame(split_summary).to_string(index=False))

print('\nTest Partition Scenario Distribution (All 11 Classes Represented):')
test_scenarios = df_model[df_model['temporal_split'] == 'test']['target_multiclass'].value_counts()
print(test_scenarios.to_string())

=== TEMPORAL SPLIT PARTITION AUDIT ===
Partition  Records Share  Start Epoch  End Epoch Suspicious %  Scenario Classes
    train     7000 70.0%   1767225656 1767490648       44.91%                11
      val     1500 15.0%   1767490665 1767547993       42.13%                11
     test     1500 15.0%   1767548015 1767609703       37.40%                11

Test Partition Scenario Distribution (All 11 Classes Represented):
target_multiclass
normal                  841
transaction_burst       143
rapid_multihop          116
peeling_chain           111
benign_high_volume       98
coordinated_activity     74
high_fan_in              34
high_fan_out             29
mixing_like              22
amount_anomaly           16
temporal_anomaly         16


In [5]:
# Feature Taxonomy and Redundancy Verification
print('=== CORRECTED FEATURE TAXONOMY AUDIT ===')
print(f"1. Primary Canonical Predictive Features (Total = {len(manifest['canonical_feature_names'])}):")
print(f"   - Canonical Numeric Features:        {manifest['dataset_summary']['canonical_numeric_features_count']}")
print(f"   - Canonical Categorical Features:    {manifest['dataset_summary']['canonical_categorical_features_count']} (net_country, net_asn)")
print(f"2. Optional Graph Ablation Features:    {manifest['dataset_summary']['optional_ablation_features_count']} (graph_fan_in/out, graph_unique_in/out_addrs)")
print(f"3. Identifiers & Entity Grouping Keys:  {manifest['dataset_summary']['identifiers_and_grouping_count']} (transaction_id, timestamp_epoch_sec, hist_cluster_id)")

# Verification: hist_cluster_id is NOT a model feature
assert 'hist_cluster_id' not in manifest['canonical_feature_names'], 'CRITICAL: hist_cluster_id must not be a model feature!'
print('[+] Verified: hist_cluster_id is strictly classified as an entity grouping key, NOT an ML feature.')

# Verification: 4 graph duplicates are NOT canonical model features
for dup in ['graph_fan_in', 'graph_fan_out', 'graph_unique_in_addrs', 'graph_unique_out_addrs']:
    assert dup not in manifest['canonical_feature_names'], f'CRITICAL: {dup} must not be in canonical_feature_names!'
print('[+] Verified: 4 graph duplicates are excluded from canonical features (retained for ablation).')

# Construct primary predictive matrix X and assert shape
X = df_model[manifest['canonical_feature_names']]
print(f'[+] Primary Feature Matrix X Shape: {X.shape[0]:,} rows x {X.shape[1]} columns (46 features).')

=== CORRECTED FEATURE TAXONOMY AUDIT ===
1. Primary Canonical Predictive Features (Total = 46):
   - Canonical Numeric Features:        44
   - Canonical Categorical Features:    2 (net_country, net_asn)
2. Optional Graph Ablation Features:    4 (graph_fan_in/out, graph_unique_in/out_addrs)
3. Identifiers & Entity Grouping Keys:  3 (transaction_id, timestamp_epoch_sec, hist_cluster_id)
[+] Verified: hist_cluster_id is strictly classified as an entity grouping key, NOT an ML feature.
[+] Verified: 4 graph duplicates are excluded from canonical features (retained for ablation).
[+] Primary Feature Matrix X Shape: 10,000 rows x 46 columns (46 features).


In [6]:
# Preprocessing Blueprint Summary for Downstream Modeling (Phase 2.5B / 2.6)
print('=== PREPROCESSING & MODELING BLUEPRINT (Phase 2.5B / 2.6) ===')
print('1. Numerical Preprocessing (44 features):')
print('   - Fit RobustScaler / StandardScaler strictly on df_model[temporal_split == "train"].')
print('   - Transform "val" and "test" splits without refitting to prevent leakage.')
print('2. Categorical Encoding (2 features):')
print('   - net_country: OneHotEncoder (16 countries, low cardinality).')
print('   - net_asn: FrequencyEncoder or TargetEncoder with out-of-fold regularization on train.')
print('3. High-Cardinality Grouping Key:')
print('   - hist_cluster_id: Retained for behavioral entity tracking, investigative pivot, and cluster risk aggregation.')
print('4. Ablation Roadmap:')
print('   - Baseline Model: 46 Canonical Features.')
print('   - Ablation Variant: +4 Optional Graph Duplicates (to prove redundancy invariance).')

print('\n[+] Modeling dataset audit complete. ZERO ML models trained.')

=== PREPROCESSING & MODELING BLUEPRINT (Phase 2.5B / 2.6) ===
1. Numerical Preprocessing (44 features):
   - Fit RobustScaler / StandardScaler strictly on df_model[temporal_split == "train"].
   - Transform "val" and "test" splits without refitting to prevent leakage.
2. Categorical Encoding (2 features):
   - net_country: OneHotEncoder (16 countries, low cardinality).
   - net_asn: FrequencyEncoder or TargetEncoder with out-of-fold regularization on train.
3. High-Cardinality Grouping Key:
   - hist_cluster_id: Retained for behavioral entity tracking, investigative pivot, and cluster risk aggregation.
4. Ablation Roadmap:
   - Baseline Model: 46 Canonical Features.
   - Ablation Variant: +4 Optional Graph Duplicates (to prove redundancy invariance).

[+] Modeling dataset audit complete. ZERO ML models trained.
